# Week 15, Day 6: Forge: Week 15 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w015/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import lgamma
print("1. e^-2 =", round(np.exp(-2), 4))
print("2. 10 / sqrt(400) =", 10 / np.sqrt(400))
print("3. mu + sigma^2 mu^2 =", 10 + 0.2 * 10 ** 2)

### Exercise: Part A: exponential or gamma?

Compute `cv`. Fit the exponential rate r = 1/mean and its log-likelihood `ll_exp` = Σ log(r e^(−r t)). Fit a gamma by moments, shape `k_hat` = mean²/var and scale θ = var/mean, and compute `ll_gamma` = Σ [(k − 1) log t − t/θ − k log θ − log Γ(k)] (use `lgamma`).

In [ ]:
rng = np.random.default_rng(1998)
isi = 0.002 + rng.gamma(3.0, 0.048 / 3, 3000)        # seconds; includes a 2 ms refractory period

cv = ...
r = 1 / isi.mean()
ll_exp = ...
k_hat = ...
theta = isi.var() / isi.mean()
ll_gamma = ...
print(f"CV {cv:.3f};  log-likelihoods: exponential {ll_exp:.1f}, gamma (k = {k_hat:.2f}) {ll_gamma:.1f}")
t = np.linspace(0.0005, 0.2, 400)
plt.hist(isi, 80, density=True, alpha=0.5); plt.plot(t, r * np.exp(-r * t), label="exponential")
plt.plot(t, np.exp((k_hat - 1) * np.log(t) - t / theta - k_hat * np.log(theta) - lgamma(k_hat)), label="gamma"); plt.legend(); plt.xlabel("interval (s)"); plt.show()

In [ ]:
import numpy as np
def test_cv():
    "the coefficient of variation is well below 1"
    assert abs(cv - isi.std() / isi.mean()) < 1e-12 and cv < 0.8

def test_ll():
    "the gamma model's log-likelihood beats the exponential's by more than 100"
    assert ll_gamma > ll_exp + 100

def test_shape():
    "the fitted gamma shape is 1 / CV squared"
    assert abs(k_hat - 1 / cv ** 2) < 1e-9

test_cv()
test_ll()
test_shape()
print("All checks passed")

### Exercise: Part B: a biased parabola

For each calcium scale in `scales`, set the site probabilities to `base * scale` (capped at 0.97), simulate 2,000 trials with quantal size 0.4 mV, and record the mean and variance. Fit the parabola as on Day 2 and store `n_hat_h` and `q_hat_h`.

In [ ]:
q, base = 0.4, np.array([0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.6, 0.8])
scales = [0.1, 0.3, 0.6, 1.0, 1.4, 1.8]
means_h, vars_h = [], []
for s in scales:
    p_sites = np.minimum(base * s, 0.97)
    amp = q * (rng.random((2000, 8)) < p_sites).sum(axis=1)
    ...
means_h, vars_h = np.array(means_h), np.array(vars_h)
a_coef, b_coef = np.linalg.lstsq(np.column_stack([means_h, means_h ** 2]), vars_h, rcond=None)[0]
q_hat_h, n_hat_h = ..., ...
print(f"heterogeneous sites: estimated n = {n_hat_h:.2f} (true 8), q = {q_hat_h:.3f} mV (true 0.4)")

In [ ]:
import numpy as np
def test_points():
    "six (mean, variance) points were measured"
    assert len(means_h) == 6 and len(vars_h) == 6

def test_bias():
    "with heterogeneous probabilities the fit underestimates both n and q"
    assert n_hat_h < 7.6 and q_hat_h < 0.38

test_points()
test_bias()
print("All checks passed")

### Exercise: Part C: the pooling limit

For N in `pool_sizes`, simulate 20,000 trials of each kind, with the stimulus chosen at random. Decide 'stimulus 2' when the pooled count exceeds 11N. Store the accuracies in `acc_indep` and `acc_shared`. Then compute the predicted ceiling for large N: the shared gain alone gives d′ = 2 / (0.05 ** 0.5 × 11), and accuracy Φ(d′/2).

In [ ]:
from math import erf, sqrt
def Phi(z): return 0.5 * (1 + erf(z / sqrt(2)))
pool_sizes = [1, 10, 100, 1000]
sigma2, T = 0.05, 20000

def accuracy(N, shared):
    s = rng.integers(0, 2, T)                         # which stimulus
    mu = np.where(s == 1, 12.0, 10.0)
    g = rng.gamma(1 / sigma2, sigma2, T) if shared else np.ones(T)
    pooled = ...                                      # one Poisson draw per trial (superposition)
    return ...

acc_indep = [accuracy(N, False) for N in pool_sizes]
acc_shared = [accuracy(N, True) for N in pool_sizes]
ceiling = ...
print("independent:", np.round(acc_indep, 3), "  shared gain:", np.round(acc_shared, 3), f"  predicted ceiling {ceiling:.3f}")
plt.semilogx(pool_sizes, acc_indep, "o-", label="independent"); plt.semilogx(pool_sizes, acc_shared, "o-", label="shared gain")
plt.axhline(ceiling, ls="--", c="k"); plt.xlabel("neurons pooled"); plt.ylabel("accuracy"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_independent():
    "with independent noise, accuracy approaches 1 for a pool of 1000"
    assert acc_indep[-1] > 0.99 and acc_indep[0] < 0.7

def test_shared():
    "with shared gain noise, accuracy stays below 0.7 even for 1000 neurons"
    assert max(acc_shared) < 0.7

def test_ceiling():
    "the Monte Carlo ceiling agrees with the Gaussian prediction Phi(d' / 2) to within 0.03"
    assert abs(acc_shared[-1] - ceiling) < 0.03

test_independent()
test_shared()
test_ceiling()
print("All checks passed")